# 🧠 Deterministic Tool Execution

This notebook explains **how to design tool execution so that
LLMs never directly control reality**.

You will learn:
- Why LLMs must never execute tools directly
- What “deterministic execution” actually means
- The correct control boundary between LLM and system
- How retries, validation, and idempotency work together
- Why most agent systems fail dangerously here

📌 Core rule:
> LLMs decide.
> Systems execute.


## 1. The Core Risk

LLMs are:
- probabilistic
- non-deterministic
- hallucination-prone

Tools are:
- real
- state-changing
- irreversible

Connecting them directly is dangerous.


## 2. Deterministic Execution

Deterministic execution means:

- same input → same outcome
- execution controlled by code
- validation before action
- predictable failure modes

The LLM never executes logic.
It only proposes actions.


## 3. Correct Control Boundary

LLM responsibilities:
- interpret user intent
- select tool
- provide structured arguments

System responsibilities:
- validate inputs
- enforce rules
- execute tools
- handle errors


## 4. Anti-Pattern

❌ LLM calls APIs directly  
❌ LLM constructs SQL and executes it  
❌ LLM decides retries  
❌ LLM handles errors  

This turns probabilistic text into control flow.


## 5. Deterministic Tool Execution Pipeline

```text
User Request
   ↓
LLM (decision + arguments)
   ↓
Schema Validation
   ↓
Policy Validation
   ↓
Deterministic Tool Execution
   ↓
Result Logging
   ↓
LLM (explanation only)
```


## 6. Example Tool

Task: "Create a support ticket"

The tool definition is strict.
The LLM must conform to it.


In [ ]:
create_ticket_schema = {
    "name": "create_ticket",
    "description": "Create a customer support ticket",
    "parameters": {
        "type": "object",
        "properties": {
            "customer_id": {"type": "string"},
            "issue_type": {
                "type": "string",
                "enum": ["billing", "technical", "account"]
            },
            "priority": {
                "type": "string",
                "enum": ["low", "medium", "high"]
            },
            "description": {"type": "string"}
        },
        "required": ["customer_id", "issue_type", "priority", "description"]
    }
}


## 7. LLM Responsibility

The LLM may ONLY:
- select `create_ticket`
- populate arguments
- explain results afterward

It may NOT:
- decide execution timing
- retry failures
- modify system state


## 8. Validation

Before execution:
- schema validation
- permission checks
- rate limits
- business rules

Invalid requests:
> are rejected, not “fixed” by the LLM


In [ ]:
def create_ticket(args):
    # deterministic behavior
    ticket_id = f"TICKET-{hash(str(args)) % 100000}"
    return {
        "ticket_id": ticket_id,
        "status": "created"
    }


## 10. Error Handling

LLMs must never:
- catch exceptions
- decide retries
- interpret stack traces

Error handling belongs to:
> deterministic system logic


## 11. Retry Logic

Retries should be:
- bounded
- logged
- idempotent

LLMs should never say:
“Try again.”


## 12. Idempotency

Idempotent tools ensure:
- duplicate requests do not duplicate effects
- retries are safe
- crashes do not corrupt state

This is mandatory for:
- payments
- tickets
- database updates


## 13. Observability

Always log:
- tool name
- arguments
- validation outcome
- execution result
- timestamps

If it isn’t logged,
it didn’t happen safely.


## 14. Determinism vs Intelligence

Deterministic execution:
- reduces hallucination
- prevents runaway behavior
- enables compliance

This is not a loss of intelligence.
It is a gain in trust.


## 15. Failure Modes

❌ Letting LLM modify arguments post-validation  
❌ Allowing partial execution  
❌ Silent retries  
❌ Tool logic in prompts  
❌ No idempotency keys  


## Final Mental Lock

LLMs propose.
Systems dispose.

If an LLM can execute,
the system is unsafe.


## Self-Check

You understand this notebook if you can explain:

- Why LLMs must not execute tools
- Where determinism comes from
- Why validation precedes execution
- Why retries and idempotency are mandatory


Most GenAI disasters happen
after the model stops talking
and starts acting.

Deterministic execution
is how you prevent that.
